In [1]:
import pandas as pd

import re

from typing import Tuple
from united_board_accessment.reference_data import COUNTRIES, COUNTRY_NAMES, INSTITUTIONS, INSTITUTION_ALIASES

In [2]:
applications_df = pd.read_csv("../../data/applications.csv", keep_default_na=False)
existing_records_df = pd.read_csv("../../data/existing_records.csv", keep_default_na=False)

In [3]:
applications_df.head(n=12)

,id,submitted,full_name,email,institution,country,position,endorsement
0,1,05 Jan 2027,Marisol Tavera,m.tavera@su.example,Silliman University,Philippines,Associate Professor,Yes
1,2,05 Jan 2027,rohan varghese,rohan.varghese@mcc.example,Madras Christian College,India,Dean,Yes
2,3,06 Jan 2027,Rohan Varghese,rohan.varghese@mcc.example,Madras Christian Coll.,India,Dean,Yes
3,4,06 Jan 2027,Sekar Wulandari,sekar.w@ukdw.example,UKDW,Indonesia,Lecturer,No
4,5,07 Jan 2027,Liza Camposano,l.camposano@su.example,Silliman Univ.,PH,Lecturer,Yes
5,6,08 Jan 2027,Kavitha Mohandas,kavitha.m@ldc.example,Lady Doak College,India,Head of Department,Yes
6,7,08 Jan 2027,Dimas Prakoso,dimas.prakoso@uksw,Satya Wacana Christian University,Indonesia,Lecturer,Yes
7,8,09 Jan 2027,Nattaya Rungrot,nattaya.r@payap.example,Payap University,Thailand,,Yes
8,9,09 Jan 2027,Rafael Mendiola,r.mendiola@cpu.example,Central Philipine University,Philippines,Associate Professor,Yes
9,10,10 Jan 2027,Pimchanok Thawee,pimchanok.t@payap.example,Payap University,Thailand,Lecturer,Yes


In [4]:
existing_records_df.head()

,full_name,email,institution,country
0,Rafael Mendiola,r.mendiola@cpu.example,Central Philippine University,Philippines
1,Kavitha Mohandas,kavitha.mohandas@personalmail.example,Lady Doak College,India
2,Tiur Situmorang,tiur.situmorang@ukdw.example,Duta Wacana Christian University,Indonesia
3,Marisol Tavera,marisol.tavera@slu.example,Saint Louis University,Philippines


### normalize

In [5]:
# trims the end and collapse repeated spaces
def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

In [6]:
# lowercase and turn run of spaces/punctuation into one underscore
def to_key(s: str) -> str:
    return re.sub(r"[\W_]+", "_", s.lower()).strip("_")

In [7]:
test_str_li = [" Madras   Christian  Coll. ", "PH", "Mr.Nyein Chan Soe"]

In [8]:
for t_str in test_str_li:
    ct = clean_text(t_str)
    k = to_key(ct)
    print(ct, k)

Madras Christian Coll. madras_christian_coll
PH ph
Mr.Nyein Chan Soe mr_nyein_chan_soe


In [9]:
COUNTRY_CODE_LOOKUP = {to_key(k): v for k, v in COUNTRIES.items()}
COUNTRY_NAME_LOOKUP = {to_key(k): v for k, v in COUNTRY_NAMES.items()}
INSTITUTION_LOOKUP  = {to_key(i): i for i in INSTITUTIONS}
INSTITUTION_LOOKUP.update({to_key(k): v for k, v in INSTITUTION_ALIASES.items()})

In [10]:
def norm_country(s: str) -> Tuple[str, bool]:
    c = clean_text(s)
    k = to_key(c)
    if k in COUNTRY_CODE_LOOKUP:
        return (COUNTRY_CODE_LOOKUP[k], True)
    if k in COUNTRY_NAME_LOOKUP:
        return (COUNTRY_NAME_LOOKUP[k], True)
    return (c, False)

In [11]:
norm_country("India")

('India', True)

In [12]:
def norm_institution(s: str) -> Tuple[str, bool]:
    c = clean_text(s)
    k = to_key(c)
    if k in INSTITUTION_LOOKUP:
        return (INSTITUTION_LOOKUP[k], True)
    return (c, False)

In [13]:
norm_institution("UKDW")

('Duta Wacana Christian University', True)

In [14]:
def norm_name(s: str) -> str:
    c = clean_text(s)
    if c.isupper() or c.islower():
        return c.title()
    return c

In [15]:
norm_name("NYEIN CHAN")

'Nyein Chan'

In [16]:
# just cleaning up and lower no validation
def norm_email(s: str) -> str:
    c = clean_text(s).lower()
    return c

In [17]:
norm_email("NyeinchanSoe@gmail.com")

'nyeinchansoe@gmail.com'

In [18]:
def normalise_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["full_name"] = out["full_name"].map(norm_name)
    out["email"] = out["email"].map(norm_email)

    res = out["country"].map(norm_country)
    out["country"], out["country_known"] = res.str[0], res.str[1]

    res = out["institution"].map(norm_institution)
    out["institution"], out["institution_known"] = res.str[0], res.str[1]

    if "position" in out.columns:
        out["position"] = out["position"].map(clean_text)
    return out

In [19]:
norm_apps_df = normalise_df(applications_df)
norm_existing_df = normalise_df(existing_records_df)

In [20]:
norm_apps_df.head()

,id,submitted,full_name,email,institution,country,position,endorsement,country_known,institution_known
0,1,05 Jan 2027,Marisol Tavera,m.tavera@su.example,Silliman University,Philippines,Associate Professor,Yes,True,True
1,2,05 Jan 2027,Rohan Varghese,rohan.varghese@mcc.example,Madras Christian College,India,Dean,Yes,True,True
2,3,06 Jan 2027,Rohan Varghese,rohan.varghese@mcc.example,Madras Christian College,India,Dean,Yes,True,True
3,4,06 Jan 2027,Sekar Wulandari,sekar.w@ukdw.example,Duta Wacana Christian University,Indonesia,Lecturer,No,True,True
4,5,07 Jan 2027,Liza Camposano,l.camposano@su.example,Silliman University,Philippines,Lecturer,Yes,True,True


In [21]:
norm_existing_df.head()

,full_name,email,institution,country,country_known,institution_known
0,Rafael Mendiola,r.mendiola@cpu.example,Central Philippine University,Philippines,True,True
1,Kavitha Mohandas,kavitha.mohandas@personalmail.example,Lady Doak College,India,True,True
2,Tiur Situmorang,tiur.situmorang@ukdw.example,Duta Wacana Christian University,Indonesia,True,True
3,Marisol Tavera,marisol.tavera@slu.example,Saint Louis University,Philippines,True,True


### Validation (critical fields)

A critical field should not be empty. Otherwise it needs REVIEW

In [22]:
REQUIRED = ["full_name", "email", "institution", "country", "position", "endorsement"]

def missing_fields(row: pd.Series) -> list[str]:
    return [
        col for col in REQUIRED
        if pd.isna(row[col]) or str(row[col]).strip() == ""
    ]

In [23]:
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def is_valid_email(s: str) -> bool:
    return bool(EMAIL_RE.match(s))

In [24]:
norm_apps_df["missing"] = norm_apps_df.apply(missing_fields, axis=1)
norm_apps_df["email_valid"] = norm_apps_df["email"].map(is_valid_email)

In [25]:
norm_apps_df.loc[norm_apps_df["id"] == 11].missing

10    [full_name]
Name: missing, dtype: object

In [26]:
norm_apps_df.loc[~norm_apps_df["email_valid"], ["id", "email"]]

,id,email
6,7,dimas.prakoso@uksw


### datetime format

`05 Jan 2027` => (day, abbreviated month, year)\
To ISO: YYYY-MM-DD \
`05 Jan 2027` => `2027-01-05`

In [27]:
# ISO
norm_apps_df["submitted_dt"] = pd.to_datetime(
    norm_apps_df["submitted"].map(clean_text),
    format="%d %b %Y",
    errors="coerce",
)
norm_apps_df["date_valid"] = norm_apps_df["submitted_dt"].notna()

### dedupe

In [28]:
text_cols = ["full_name", "email", "institution", "country", "position", "endorsement"]

# set aside rows with a bad date
held_df = norm_apps_df[~norm_apps_df["date_valid"]]

# sort by date
# drop stale cols: because I will recompute these cols after merged
work_df = (
    norm_apps_df[norm_apps_df["date_valid"]]
    .drop(columns=["missing", "email_valid", "country_known", "institution_known"])
    .sort_values("submitted_dt")
    .copy()
)

take the latest value that isn’t blank\
so turn blanks into missing

In [29]:
text_cols = ["full_name", "email", "institution", "country", "position", "endorsement"]
work_df[text_cols] = work_df[text_cols].replace("", pd.NA)

In [30]:
print(work_df[text_cols].isna().sum())

full_name      1
email          0
institution    0
country        0
position       1
endorsement    0
dtype: int64


collapse duplicate emails

In [31]:
# take the last value that isn't missing
deduped_df = work_df.groupby("email", as_index=False, dropna=False).last()

In [32]:
print(deduped_df.shape)
print(deduped_df[deduped_df["email"] == "rohan.varghese@mcc.example"])

(11, 10)
                        email  id    submitted       full_name  \
8  rohan.varghese@mcc.example   3  06 Jan 2027  Rohan Varghese   

                institution country position endorsement submitted_dt  \
8  Madras Christian College   India     Dean         Yes   2027-01-06   

   date_valid  
8        True  


In [33]:
deduped_df[text_cols] = deduped_df[text_cols].fillna("")

In [34]:
print(deduped_df[text_cols].isna().sum().sum())
print(deduped_df.loc[deduped_df["id"].isin([8, 11]), ["id", "full_name", "position"]])

0
   id        full_name  position
1  11                   Lecturer
5   8  Nattaya Rungrot          


remember which submissions were merged

In [35]:
merged_ids = work_df.groupby("email", dropna=False)["id"].apply(list).rename("merged_ids")
deduped_df = deduped_df.merge(merged_ids, on="email")

In [36]:
print(deduped_df[["id", "email", "merged_ids"]])

    id                       email merged_ids
0    7          dimas.prakoso@uksw        [7]
1   11         j.koshy@mcc.example       [11]
2    6       kavitha.m@ldc.example        [6]
3    5      l.camposano@su.example        [5]
4    1         m.tavera@su.example        [1]
5    8     nattaya.r@payap.example        [8]
6   10   pimchanok.t@payap.example       [10]
7    9      r.mendiola@cpu.example        [9]
8    3  rohan.varghese@mcc.example     [2, 3]
9    4        sekar.w@ukdw.example        [4]
10  12         tiur.s@ukdw.example       [12]


In [37]:
deduped_df = normalise_df(deduped_df)

In [38]:
print(deduped_df.shape)
print(deduped_df[["country_known", "institution_known"]].all())

(11, 13)
country_known        True
institution_known    True
dtype: bool


In [39]:
deduped_df["email_valid"] = deduped_df["email"].map(is_valid_email)
deduped_df["missing"] = deduped_df.apply(missing_fields, axis=1)

In [40]:
problems = deduped_df[(~deduped_df["email_valid"]) | (deduped_df["missing"].str.len() > 0)]
print(problems[["id", "email_valid", "missing"]])

   id  email_valid      missing
0   7        False           []
1  11         True  [full_name]
5   8         True   [position]


In [41]:
existing = norm_existing_df.reset_index(drop=True)
existing["existing_id"] = existing.index

email_to_existing = dict(zip(existing["email"], existing["existing_id"]))
deduped_df["match_email"] = deduped_df["email"].map(email_to_existing)

In [42]:
print(deduped_df.loc[deduped_df["match_email"].notna(), ["id", "email", "match_email"]])

   id                   email  match_email
7   9  r.mendiola@cpu.example          0.0


### Issue Logging

In [43]:
issue_log = []

def log_issue(app_id, applicant, issue, action, follow_up=""):
    issue_log.append({
        "id": app_id, "applicant": applicant,
        "issue": issue, "action": action, "follow_up": follow_up,
    })

In [44]:
for _, r in deduped_df.iterrows():
    if len(r["merged_ids"]) > 1:
        log_issue(r["id"], r["full_name"],
                  f"Duplicate submission (ids {r['merged_ids']})",
                  f"Merged into id {r['id']}; latest submission kept")

In [45]:
print(pd.DataFrame(issue_log))

   id       applicant                              issue  \
0   3  Rohan Varghese  Duplicate submission (ids [2, 3])   

                                     action follow_up  
0  Merged into id 3; latest submission kept            


In [46]:
FIELD_LABELS = {
    "full_name":   ("Name case fixed",     "Standardized"),
    "institution": ("Institution variant", "Mapped to master name"),
    "country":     ("Country code",        "Expanded"),
}

for field, (issue, action) in FIELD_LABELS.items():
    changed = applications_df[field] != norm_apps_df[field]
    for i in applications_df.index[changed]:
        raw, new = applications_df.at[i, field], norm_apps_df.at[i, field]
        log_issue(applications_df.at[i, "id"], norm_apps_df.at[i, "full_name"],
                  f"{issue}: '{raw}' -> '{new}'", action)

In [47]:
log_df = pd.DataFrame(issue_log)
print(len(log_df))
print(log_df[["id", "applicant", "issue"]])

7
   id        applicant                                              issue
0   3   Rohan Varghese                  Duplicate submission (ids [2, 3])
1   2   Rohan Varghese  Name case fixed: 'rohan varghese' -> 'Rohan Va...
2   3   Rohan Varghese  Institution variant: 'Madras Christian Coll.' ...
3   4  Sekar Wulandari  Institution variant: 'UKDW' -> 'Duta Wacana Ch...
4   5   Liza Camposano  Institution variant: 'Silliman Univ.' -> 'Sill...
5   9  Rafael Mendiola  Institution variant: 'Central Philipine Univer...
6   5   Liza Camposano                Country code: 'PH' -> 'Philippines'


In [48]:
for _, r in deduped_df.iterrows():
    for dropped_id in [i for i in r["merged_ids"] if i != r["id"]]:
        log_issue(dropped_id, r["full_name"],
                  f"Duplicate of id {r['id']}",
                  "Excluded from import (kept the later submission)")

In [49]:
log_df = pd.DataFrame(issue_log)
print(len(log_df))
print(log_df.loc[log_df["id"] == 2, ["id", "issue", "action"]])

8
   id                                              issue  \
1   2  Name case fixed: 'rohan varghese' -> 'Rohan Va...   
7   2                                  Duplicate of id 3   

                                             action  
1                                      Standardized  
7  Excluded from import (kept the later submission)  


In [50]:
for _, r in deduped_df[~deduped_df["email_valid"]].iterrows():
    log_issue(r["id"], r["full_name"],
              f"Invalid email format: '{r['email']}'",
              "Held for review (not imported)",
              "Contact applicant to confirm email address")

In [51]:
log_df = pd.DataFrame(issue_log)
print(len(log_df))
print(log_df.loc[log_df["id"] == 7, ["id", "issue", "action", "follow_up"]])

9
   id                                       issue  \
8   7  Invalid email format: 'dimas.prakoso@uksw'   

                           action                                   follow_up  
8  Held for review (not imported)  Contact applicant to confirm email address  


In [52]:
for _, r in deduped_df[deduped_df["missing"].str.len() > 0].iterrows():
    for field in r["missing"]:
        log_issue(r["id"], r["full_name"] or "(no name given)",
                  f"Missing required field: {field}",
                  "Held for review (not imported)",
                  f"Request {field.replace('_', ' ')} from applicant")

In [53]:
log_df = pd.DataFrame(issue_log)
print(len(log_df))
print(log_df.loc[log_df["id"].isin([8, 11]), ["id", "applicant", "issue", "follow_up"]])

11
    id        applicant                              issue  \
9   11  (no name given)  Missing required field: full_name   
10   8  Nattaya Rungrot   Missing required field: position   

                           follow_up  
9   Request full name from applicant  
10   Request position from applicant  


In [54]:
for col, label in [("institution_known", "institution"), ("country_known", "country")]:
    for _, r in deduped_df[~deduped_df[col]].iterrows():
        log_issue(r["id"], r["full_name"] or "(no name given)",
                  f"Unrecognised {label}: '{r[label]}'",
                  "Held for review (not imported)",
                  f"Confirm {label}; add alias if it is a known variant")

In [55]:
print(len(pd.DataFrame(issue_log)))

11


In [56]:
existing = norm_existing_df.reset_index(drop=True)
existing["existing_id"] = existing.index

email_to_existing = dict(zip(existing["email"], existing["existing_id"]))
deduped_df["match_email"] = deduped_df["email"].map(email_to_existing)

In [57]:
print(deduped_df.loc[deduped_df["match_email"].notna(), ["id", "full_name", "email", "match_email"]])

   id        full_name                   email  match_email
7   9  Rafael Mendiola  r.mendiola@cpu.example          0.0


In [58]:
def person_key(df):
    return (df["full_name"].map(to_key) + "|" + df["institution"].map(to_key)
            + "|" + df["country"].map(to_key))

existing["person_key"] = person_key(existing)
deduped_df["person_key"] = person_key(deduped_df)

key_to_existing = dict(zip(existing["person_key"], existing["existing_id"]))
deduped_df["match_person"] = deduped_df["person_key"].map(key_to_existing)

In [59]:
print(deduped_df.loc[deduped_df["match_person"].notna(), ["id", "full_name", "match_email", "match_person"]])

    id         full_name  match_email  match_person
2    6  Kavitha Mohandas          NaN           1.0
7    9   Rafael Mendiola          0.0           0.0
10  12   Tiur Situmorang          NaN           2.0


### Pipeline

The finished logic lives in `pipeline.py`. The cells below only import it and show the results.\
The exploration cells above are kept as working notes.

In [60]:
from united_board_accessment import pipeline

apps_raw, existing_raw = pipeline.load()
import_ready, issues = pipeline.run(apps_raw, existing_raw)

In [61]:
import_ready["status"].value_counts()

status
Needs review              5
New                       3
Update existing record    3
Name: count, dtype: int64

In [62]:
import_ready

,submission_ids,submitted,full_name,email,institution,country,position,endorsement,status,reason,matched_existing
id,,,,,,,,,,,
1,1,2027-01-05,Marisol Tavera,m.tavera@su.example,Silliman University,Philippines,Associate Professor,Yes,Needs review,Name-only match to existing record Marisol Tav...,Marisol Tavera <marisol.tavera@slu.example>
3,"2, 3",2027-01-06,Rohan Varghese,rohan.varghese@mcc.example,Madras Christian College,India,Dean,Yes,New,No matching existing record,
4,4,2027-01-06,Sekar Wulandari,sekar.w@ukdw.example,Duta Wacana Christian University,Indonesia,Lecturer,No,Needs review,Endorsement = No,
5,5,2027-01-07,Liza Camposano,l.camposano@su.example,Silliman University,Philippines,Lecturer,Yes,New,No matching existing record,
6,6,2027-01-08,Kavitha Mohandas,kavitha.m@ldc.example,Lady Doak College,India,Head of Department,Yes,Update existing record,Matched existing record by name + institution ...,Kavitha Mohandas <kavitha.mohandas@personalmai...
7,7,2027-01-08,Dimas Prakoso,dimas.prakoso@uksw,Satya Wacana Christian University,Indonesia,Lecturer,Yes,Needs review,Invalid email format: 'dimas.prakoso@uksw',
8,8,2027-01-09,Nattaya Rungrot,nattaya.r@payap.example,Payap University,Thailand,,Yes,Needs review,Missing required field: position,
9,9,2027-01-09,Rafael Mendiola,r.mendiola@cpu.example,Central Philippine University,Philippines,Associate Professor,Yes,Update existing record,Matched existing record by email,Rafael Mendiola <r.mendiola@cpu.example>
10,10,2027-01-10,Pimchanok Thawee,pimchanok.t@payap.example,Payap University,Thailand,Lecturer,Yes,New,No matching existing record,


In [63]:
issues

,id,applicant,issue,action,follow_up
0,1,Marisol Tavera,Name-only match to existing record Marisol Tav...,Needs review,Confirm whether same person
1,2,Rohan Varghese,Name case: 'rohan varghese' -> 'Rohan Varghese',Standardized,
2,2,Rohan Varghese,Duplicate of id 3,Excluded from import,
3,3,Rohan Varghese,Institution variant: 'Madras Christian Coll.' ...,Mapped to master name,
4,3,Rohan Varghese,"Duplicate submission (ids [2, 3])",Merged; latest kept,
5,4,Sekar Wulandari,Institution variant: 'UKDW' -> 'Duta Wacana Ch...,Mapped to master name,
6,4,Sekar Wulandari,Endorsement = No,Needs review,Staff to decide eligibility
7,5,Liza Camposano,Institution variant: 'Silliman Univ.' -> 'Sill...,Mapped to master name,
8,5,Liza Camposano,Country code: 'PH' -> 'Philippines',Expanded,
9,6,Kavitha Mohandas,Matched existing record by name + institution ...,Update existing record; position and endorseme...,


In [64]:
print(len(pipeline.NOTE.split()), "words\n")
print(pipeline.NOTE)

147 words

Most problems in this batch are data-entry problems, and the proposed workflow stops them at the form.

Dropdown lists for country and institution prevent variants and typos such as "PH", "Silliman Univ.", "Madras Christian Coll.", "UKDW" and "Central Philipine University". Email format validation and required fields prevent "dimas.prakoso@uksw", the blank name (id 11) and the blank position (id 8): the form cannot be submitted until they are fixed. Duplicate detection on email catches Rohan Varghese's second submission and keeps the latest. Each submission is then matched against the database automatically. A match on email, or on name, institution and country together, becomes an update. Anything uncertain goes to a review queue and is not imported.

Automation cannot settle genuine conflicts or eligibility. Marisol Tavera's name matches a record at a different institution, and Sekar Wulandari's endorsement is "No". The workflow flags both, but staff make the decision.


In [65]:
# writes the two CSV files and note.txt to outputs/
pipeline.write_outputs(import_ready, issues)

PosixPath('/Users/nyeinchan/Desktop/united-board-accessment/outputs')